# Linear Regression Implementation

Linear regression:
    Uses Big Five traits to predict continuous CWB and OCB.

Algorithm idea:
    Linear regression models the outcome as a weighted linear combination of the Big Five traits.

Evaluation metrics:
    - **R²**: proportion of variance in the outcome explained by the predictors
    - **MAE**: average absolute prediction error
    - **RMSE**: prediction error that penalizes larger mistakes more heavily

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np


def run_linear_regression(df, random_state=42):

    big_five = [
        "Extraversion",
        "Agreeableness",
        "Conscientiousness",
        "Neuroticism",
        "Openness",
    ]

    outcomes = ["CWB", "OCB"]
    results = {}

    for outcome in outcomes:
        data = df[big_five + [outcome]].dropna().copy()

        X = data[big_five]
        y = data[outcome]

        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=0.2,
            random_state=random_state
        )

        model = LinearRegression()
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        metrics = {
            "algorithm": "Linear Regression",
            "outcome": outcome,
            "task": "regression",
            "MAE": mean_absolute_error(y_test, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
            "R2": r2_score(y_test, y_pred),
            "coefficients": pd.DataFrame({
                "Predictor": big_five,
                "Coefficient": model.coef_
            }),
            "intercept": model.intercept_,
            "model": model,
        }

        results[outcome] = metrics

    return results

## Import data & relavent libraries

Preprocess data to transform trait and organizational behavioral measures.

In [2]:
import json
from pathlib import Path
import pandas as pd

# Find repo root automatically
repo_root = Path.cwd().resolve()
while not (repo_root / "BFI_2_life_narative_metadata.json").exists():
    if repo_root == repo_root.parent:
        raise FileNotFoundError("Could not find BFI_2_life_narative_metadata.json")
    repo_root = repo_root.parent

# Load data
with open(repo_root / "BFI_2_life_narative_metadata.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

with open(repo_root / "BFI_2_life_narrative.json", "r", encoding="utf-8") as f:
    responses = json.load(f)

df = pd.DataFrame(responses)

# Reverse-code BFI items
reverse_items = metadata["reverse_code_items"]

for item in reverse_items:
    if item in df.columns:
        df[item] = 6 - df[item]

# Big Five trait means
traits = {
    key: value
    for key, value in metadata.items()
    if (
        isinstance(value, list)
        and value
        and isinstance(value[0], str)
        and value[0].startswith("Item")
        and not key.startswith(("Item", "Q", "CWB", "OCB"))
        and key != "reverse_code_items"
    )
}

for trait, items in traits.items():
    available_items = [item for item in items if item in df.columns]
    df[trait] = df[available_items].mean(axis=1)

# CWB and OCB means
cwb_cols = [f"CWB{i}" for i in range(1, 11) if f"CWB{i}" in df.columns]
ocb_cols = [f"OCB{i}" for i in range(1, 11) if f"OCB{i}" in df.columns]

df["CWB"] = df[cwb_cols].mean(axis=1)
df["OCB"] = df[ocb_cols].mean(axis=1)

# Targets
big_five = [
    "Extraversion",
    "Agreeableness",
    "Conscientiousness",
    "Neuroticism",
    "Openness",
]

print("✓ Data preprocessing complete")
print("Repo root:", repo_root)
print("df shape:", df.shape)
df.head()

✓ Data preprocessing complete
Repo root: /Users/cindy/cmor438_Spring2026/cmor438_Spring2026
df shape: (500, 138)


,ID,Gender,Race,Age,Q1,Q2,Q3,Q4,Q5,Q6,...,Intellectual_Curiosity,Aesthetic_Sensitivity,Creative_Imagination,Extraversion,Agreeableness,Conscientiousness,Neuroticism,Openness,CWB,OCB
0,1,Woman,White,22,"chicago, illinois; in a suburb near chicago. i...",i always worked very hard and did my best. i h...,i had multiple teachers that were influential ...,art or reading/writing. i am a really creative...,"math, because it doesn't always come as easily...",one of my heroes has always been my mom. she a...,...,4.50,4.75,5.00,2.416667,4.250000,3.083333,4.500000,4.750000,1.7,3.2
1,2,Woman,White,38,I am from VA and you? I grew up in TX and it w...,A very hardworking and a brilliant student in ...,I always liked my Mathematics teacher very muc...,I loved mathematics and biology as I loved to ...,Probably geography was my least favorite for l...,For me it was always Einstein as he was an awe...,...,3.50,3.75,4.25,4.583333,4.250000,4.250000,1.583333,3.833333,1.1,2.7
2,3,Woman,White,19,"I'm from Wichita,Kansas My life was great grow...",I was horrible in school. I was diagnosed with...,Oh yeah! I had a great teacher senior year of ...,My favorite subject in school was probably His...,I hated English. I was never good at it.,my heros were probably my parents. They were a...,...,3.50,4.25,4.50,3.916667,3.916667,2.500000,3.500000,4.083333,2.1,3.6
3,4,Man,White,21,"northbrook, IL; I grew up in the northern subu...",I was a great student. I was in Honors and AP ...,I definitely had influential teachers. My AP p...,Math because there was a definitive answer and...,History because it was so boring to me alwya,My mom for sure and brothers,...,3.50,3.00,3.25,4.666667,4.500000,3.083333,2.583333,3.250000,1.5,2.8
4,5,Woman,White,26,"I am from Denver Colorado, I grew up in Cherry...",I was a very bright student in school and ever...,My mathematics teacher was influtial in my pro...,Mathematics was my favorite subject because i ...,History was my worst subject because the class...,My parents were my heroes because they gave me...,...,3.75,3.25,3.25,2.916667,3.250000,3.583333,2.916667,3.416667,1.5,2.8


## Run linear regression model

In [6]:
run_linear_regression(df)

{'CWB': {'algorithm': 'Linear Regression',
  'outcome': 'CWB',
  'task': 'regression',
  'MAE': 0.3358722909258631,
  'RMSE': np.float64(0.4416584692249517),
  'R2': 0.18321132822705477,
  'coefficients':            Predictor  Coefficient
  0       Extraversion     0.163803
  1      Agreeableness    -0.135141
  2  Conscientiousness    -0.124212
  3        Neuroticism     0.135664
  4           Openness    -0.018444,
  'intercept': np.float64(1.6970695616074956),
  'model': LinearRegression()},
 'OCB': {'algorithm': 'Linear Regression',
  'outcome': 'OCB',
  'task': 'regression',
  'MAE': 0.6120047436391044,
  'RMSE': np.float64(0.757061009597305),
  'R2': 0.14324930041438122,
  'coefficients':            Predictor  Coefficient
  0       Extraversion     0.276176
  1      Agreeableness     0.070261
  2  Conscientiousness     0.038494
  3        Neuroticism     0.076865
  4           Openness     0.151734,
  'intercept': np.float64(0.6793508097640024),
  'model': LinearRegression()}}

### Variance & error analysis
CWB:
- R² = 0.183
- MAE = 0.336
- RMSE = 0.442
This means the Big Five traits explain around 18% of individual differences in CWB.

OCB:
- R² = 0.143
- MAE = 0.612
- RMSE = 0.757
The Big Five explain around 14% of variance in OCB.

### Coefficient analysis 
Higher CWB:
+ Extraversion (0.16)
+ Neuroticism (0.13)

Lower CWB:
- Agreeableness (-0.13)
- Conscientiousness (-0.12)

Two highest predictive traits for OCB:
- Extraversion (0.27)
- Openness (0.15)

Higher extraversion, higher neurotism, lower agreeableness and lower conscientiousness are often linked to more counterproductive behavior. Whereas higher extraversion and openness are associated with more organizational citizenship behaviors. This suggests that extraversion could be both beneficial and detrimental to workplace behaviors.